Steps taken:
1. loading tensorflow imdb dataset
2. Train-test split
3. Padding
4. build model architecture
5. compiled model
6. trained model on train test dataset
7. Evaluate
8. make predictions


In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [17]:
## load dataset

max_features=10000      ## number of words to consider as features
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)

In [18]:
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

(25000,) (25000,)
(25000,) (25000,)


In [19]:
## inspect the data
X_train[0]      ## words are represented as integers, each integer represents a word in the vocabulary.

[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 2,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 2,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 2,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 2,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 2,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 2,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5,
 144,
 30,
 5535,
 18,

In [20]:

y_train[0]      ## labels are represented as integers, 0 for negative sentiment and 1 for positive sentiment.

np.int64(1)

## padding

In [21]:
## padding words - make all sequences in the list have the same length by padding them with zeros at the beginning.
maxlen=500       ## cut texts after this number of words (among top max_features most common words)
X_train=sequence.pad_sequences(X_train,maxlen=maxlen)
X_test=sequence.pad_sequences(X_test,maxlen=maxlen)

## Training Simple Rnn

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_features, output_dim=128))
model.add(SimpleRNN(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.build(input_shape=(None, maxlen))
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

## COMPILING MODEL

In [30]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Implementing earlyStopping

In [31]:
## earlystopping
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

## Training the model

In [32]:
model.fit(
    X_train, y_train,
    epochs=10,      ## number of times the model will be trained on the entire training dataset
    batch_size=32,      ## number of samples per gradient update
    validation_split=0.2,       ### fraction of the training data to be used as validation data
    callbacks=[early_stopping]      
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 102s 160ms/step - accuracy: 0.7046 - loss: 2.0721 - val_accuracy: 0.8462 - val_loss: 0.3988
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 93s 149ms/step - accuracy: 0.8703 - loss: 0.3109 - val_accuracy: 0.8636 - val_loss: 0.3631
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 88s 141ms/step - accuracy: 0.9150 - loss: 0.2180 - val_accuracy: 0.8094 - val_loss: 0.4423
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 87s 138ms/step - accuracy: 0.9331 - loss: 0.1810 - val_accuracy: 0.8560 - val_loss: 0.3784
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 86s 137ms/step - accuracy: 0.9484 - loss: 0.1403 - val_accuracy: 0.8434 - val_loss: 0.4019
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 86s 138ms/step - accuracy: 0.9490 - loss: 0.1383 - val_accuracy: 0.8498 - val_loss: 0.4411
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 96s 153ms/step - accuracy: 0.9490 - loss: 0.1328 - val_accuracy: 0.8180 - val_loss: 0.4518


In [33]:
## saving model
model.save('movie_sentiment_analysis_model.h5')